# Experiment 1.3b-ii-A: Hessian Top-5 Eigenvector Alignment With Update

## Hypothesis

**Primary claim:** The Muon optimizer's orthogonalization step (polar decomposition of the gradient matrix) systematically steers the parameter update *away* from flat (gauge) directions of the loss landscape, as identified by the bottom eigenvectors of the Hessian. In contrast, vanilla SGD distributes its update energy more uniformly across all Hessian eigenvectors, including the flat gauge directions.

**Formal statement:** Let $H$ be the full Hessian of the loss at the current parameter point $\theta$, with eigendecomposition $H = \sum_i \lambda_i v_i v_i^T$ where $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_6$. Define the normalized projection of an update $\Delta\theta$ onto eigenvector $v_i$ as $p_i = |v_i^T \hat{\Delta\theta}|$ where $\hat{\Delta\theta} = \Delta\theta / \|\Delta\theta\|$. Then:

- **Muon prediction:** $\sum_{i=5}^{6} p_i^2(\text{Muon}) \ll \sum_{i=5}^{6} p_i^2(\text{SGD})$ (Muon avoids flat/gauge directions)
- **Muon prediction:** $\sum_{i=1}^{2} p_i^2(\text{Muon}) \gg \sum_{i=1}^{2} p_i^2(\text{SGD})$ (Muon concentrates on high-curvature/physical directions)

## Motivation

In deep linear networks $y = W_2 W_1 x$, the product $W_2 W_1$ is the physically meaningful quantity -- any invertible transformation $W_1 \to A W_1$, $W_2 \to W_2 A^{-1}$ leaves the network function unchanged. These reparameterization symmetries create **gauge directions** in parameter space: directions along which the loss is perfectly flat. The Hessian has zero (or near-zero) eigenvalues along these gauge directions.

The Muon optimizer replaces the raw gradient matrix $G$ with its polar factor $U V^T$ (from the SVD $G = U \Sigma V^T$). This orthogonalization discards magnitude information and equalizes singular values. The central question is: **does this orthogonalization have the geometric effect of rotating the update away from gauge-flat Hessian directions?**

This connects to the renormalization group (RG) interpretation of Muon: if Muon acts as a gauge-fixing procedure, it should naturally avoid moving along gauge orbits (flat directions) and concentrate its steps along physically meaningful (high-curvature) directions.

## Prior Results in This Experiment Series

| Experiment | Finding |
|---|---|
| 1.2b-i | Muon does NOT distinguish gauge from physical in Lyapunov exponent terms |
| 1.3b-i-A | Muon REVERSES the gradient-spectrum feedback loop (correlation = -0.51) |
| This (1.3b-ii-A) | Does Muon's step *direction* avoid flat Hessian directions? |

## Methodology

1. **Architecture:** A minimal 2-input, 2-hidden, 1-output deep linear network (6 total parameters: $W_1 \in \mathbb{R}^{2 \times 2}$, $W_2 \in \mathbb{R}^{1 \times 2}$).
2. **Hessian:** Compute the full $6 \times 6$ Hessian via central finite differences at selected training steps.
3. **Updates:** At each measurement step, compute both the SGD update ($-\nabla L$) and the Muon update ($-\text{ortho}(\nabla_{W_i} L)$ per weight matrix, concatenated).
4. **Projection:** Project each normalized update onto all 6 Hessian eigenvectors. Compare the squared projection mass on the top-2 (physical/high-curvature) vs. bottom-2 (gauge/flat) eigenvectors.
5. **Training:** The model is trained with plain SGD for 200 steps; measurements are taken at 10 specific steps throughout training.

## Expected Outcomes

- **If confirmed:** Muon's projections onto bottom-2 eigenvectors are consistently smaller than SGD's, with a ratio below 0.8 and occurring in at least 70% of measured steps.
- **If refuted:** Muon has *more* mass on gauge directions than SGD (ratio > 1.2).
- **If nuanced:** The picture is mixed, consistent with 1.2b-i's finding that Muon doesn't cleanly separate gauge from physical in all metrics.

## Setup: Imports

We use only NumPy for this experiment. The entire computation -- forward pass, loss, gradient, Hessian, SVD-based orthogonalization -- is done in pure NumPy. This is feasible because the network has only 6 parameters, making the full $6 \times 6$ Hessian cheap to compute via finite differences.

In [ ]:
import numpy as np
import os

## Configuration

**Network architecture:** A 2-layer deep linear network with dimensions $2 \to 2 \to 1$. This gives us $W_1 \in \mathbb{R}^{2 \times 2}$ (4 params) and $W_2 \in \mathbb{R}^{1 \times 2}$ (2 params) for a total of 6 parameters.

**Why this size?** With 6 parameters, the full Hessian is a $6 \times 6$ matrix, which we can compute exactly via finite differences. The deep linear structure guarantees the existence of gauge symmetries (rescaling between layers), so some Hessian eigenvalues should be near zero at points where the symmetry is approximate.

**Finite difference step:** $\epsilon = 10^{-4}$ balances numerical precision (too small causes floating-point noise) against accuracy (too large gives poor approximation of second derivatives).

**Measurement schedule:** We sample the Hessian at 10 points during 200 training steps, concentrating early (steps 0, 5, 10, 20, 40) where the loss landscape changes rapidly and spacing out later (steps 60, 80, 120, 160, 199).

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

In [ ]:
# ============================================================
# Configuration
# ============================================================
np.random.seed(42)
N_SAMPLES = 20
INPUT_DIM = 2
HIDDEN_DIM = 2
OUTPUT_DIM = 1
N_PARAMS = INPUT_DIM * HIDDEN_DIM + OUTPUT_DIM * HIDDEN_DIM  # 4 + 2 = 6
EPSILON = 1e-4        # finite-difference step
LR_SGD = 0.01
LR_MUON = 0.01
N_TRAINING_STEPS = 200
MEASURE_STEPS = [0, 5, 10, 20, 40, 60, 80, 120, 160, 199]

print("=== Experiment Configuration ===")
print(f"  Network architecture: {INPUT_DIM} -> {HIDDEN_DIM} -> {OUTPUT_DIM} (deep linear)")
print(f"  W1 shape: ({HIDDEN_DIM}, {INPUT_DIM}) = {INPUT_DIM * HIDDEN_DIM} params")
print(f"  W2 shape: ({OUTPUT_DIM}, {HIDDEN_DIM}) = {OUTPUT_DIM * HIDDEN_DIM} params")
print(f"  Total parameters: {N_PARAMS}")
print(f"  Hessian size: {N_PARAMS} x {N_PARAMS} = {N_PARAMS**2} entries")
print(f"  Finite-difference epsilon: {EPSILON}")
print(f"  Hessian evaluations needed per step: {N_PARAMS * (N_PARAMS + 1) // 2} (upper triangle)")
print(f"  Loss evaluations per Hessian: {4 * N_PARAMS * (N_PARAMS + 1) // 2} (4 per entry)")
print(f"  Training steps: {N_TRAINING_STEPS}, LR(SGD): {LR_SGD}, LR(Muon): {LR_MUON}")
print(f"  Measurement steps: {MEASURE_STEPS}")
print(f"  Random seed: 42")

## Dataset Generation

We generate a fixed random dataset of 20 samples with 2-dimensional input and 1-dimensional output. The data is drawn from standard normal distributions, so there is no special structure to exploit -- the network must learn an arbitrary linear mapping $\mathbb{R}^2 \to \mathbb{R}$.

Using a fixed random seed (42) ensures reproducibility. The small dataset size is intentional: with 6 parameters and 20 samples, the problem is mildly overparameterized, which ensures the Hessian has a non-trivial spectrum (some directions are more important than others for fitting the data).

In [ ]:
# Fixed dataset
X = np.random.randn(N_SAMPLES, INPUT_DIM)
Y = np.random.randn(N_SAMPLES, OUTPUT_DIM)

print(f"=== Dataset ===")
print(f"  X shape: {X.shape}  (N_SAMPLES x INPUT_DIM)")
print(f"  Y shape: {Y.shape}  (N_SAMPLES x OUTPUT_DIM)")
print(f"  X stats: mean={X.mean():.4f}, std={X.std():.4f}, range=[{X.min():.4f}, {X.max():.4f}]")
print(f"  Y stats: mean={Y.mean():.4f}, std={Y.std():.4f}, range=[{Y.min():.4f}, {Y.max():.4f}]")
print(f"  X^T X eigenvalues: {np.linalg.eigvalsh(X.T @ X)}")
print(f"    (These determine the data's contribution to Hessian curvature)")

## Helper Functions: Parameter Packing and Network Forward Pass

The network parameters live in two matrices ($W_1$, $W_2$), but for Hessian computation we need them as a single flat vector $\theta \in \mathbb{R}^6$. The `pack` and `unpack` functions convert between these representations.

**Convention:** The first 4 entries of $\theta$ correspond to $W_1$ (row-major), and the last 2 entries correspond to $W_2$. This ordering matters when interpreting which Hessian eigenvector components relate to which weight matrix.

The forward pass implements $y = W_2 W_1 x$ -- a deep linear network. Note that the effective mapping is $W_{\text{eff}} = W_2 W_1 \in \mathbb{R}^{1 \times 2}$, which has only 2 degrees of freedom. But we parameterize it with 6 parameters, so 4 directions in parameter space are redundant (gauge directions). In practice, the MSE loss partially breaks this symmetry, so we expect the Hessian to have 2 large eigenvalues (physical) and several small-but-nonzero eigenvalues.

In [ ]:
# ============================================================
# Helpers
# ============================================================
def pack(W1, W2):
    """Flatten W1 (2x2) and W2 (1x2) into a 6-vector."""
    return np.concatenate([W1.ravel(), W2.ravel()])

In [ ]:
def unpack(theta):
    """Reconstruct W1 (2x2) and W2 (1x2) from a 6-vector."""
    W1 = theta[:4].reshape(HIDDEN_DIM, INPUT_DIM)
    W2 = theta[4:].reshape(OUTPUT_DIM, HIDDEN_DIM)
    return W1, W2

In [ ]:
def forward(W1, W2, X_in):
    """y = W2 @ W1 @ x for each sample."""
    # X_in: (N, 2), W1: (2, 2), W2: (1, 2)
    # hidden: (N, 2), out: (N, 1)
    hidden = X_in @ W1.T   # (N, 2)
    out = hidden @ W2.T    # (N, 1)
    return out

In [ ]:
def loss_fn(theta):
    """MSE loss given flattened parameter vector."""
    W1, W2 = unpack(theta)
    pred = forward(W1, W2, X)
    return np.mean((pred - Y) ** 2)

## Helper Functions: Gradient and Hessian via Finite Differences

**Gradient:** We compute $\nabla L(\theta)$ via central finite differences:
$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta + \epsilon e_i) - L(\theta - \epsilon e_i)}{2\epsilon}$$
This requires $2 \times 6 = 12$ loss evaluations per gradient computation.

**Hessian:** We compute the full $6 \times 6$ Hessian via the four-point central difference formula:
$$H_{ij} = \frac{\partial^2 L}{\partial \theta_i \partial \theta_j} \approx \frac{L(\theta + \epsilon e_i + \epsilon e_j) - L(\theta + \epsilon e_i - \epsilon e_j) - L(\theta - \epsilon e_i + \epsilon e_j) + L(\theta - \epsilon e_i - \epsilon e_j)}{4\epsilon^2}$$
We exploit the symmetry $H_{ij} = H_{ji}$ to only compute the upper triangle, requiring $4 \times 21 = 84$ loss evaluations per Hessian.

**Why finite differences instead of automatic differentiation?** For 6 parameters, finite differences are perfectly adequate and avoid the need for a deep learning framework. The $O(\epsilon^2)$ error from central differences with $\epsilon = 10^{-4}$ is negligible compared to the eigenvalue magnitudes we observe.

In [ ]:
def compute_gradient(theta):
    """Gradient of loss via central finite differences (6-vector)."""
    grad = np.zeros(N_PARAMS)
    for i in range(N_PARAMS):
        e_i = np.zeros(N_PARAMS)
        e_i[i] = EPSILON
        grad[i] = (loss_fn(theta + e_i) - loss_fn(theta - e_i)) / (2 * EPSILON)
    return grad

In [ ]:
def compute_hessian(theta):
    """Full 6x6 Hessian via central finite differences."""
    H = np.zeros((N_PARAMS, N_PARAMS))
    for i in range(N_PARAMS):
        for j in range(i, N_PARAMS):
            e_i = np.zeros(N_PARAMS)
            e_j = np.zeros(N_PARAMS)
            e_i[i] = EPSILON
            e_j[j] = EPSILON
            fpp = loss_fn(theta + e_i + e_j)
            fpm = loss_fn(theta + e_i - e_j)
            fmp = loss_fn(theta - e_i + e_j)
            fmm = loss_fn(theta - e_i - e_j)
            H[i, j] = (fpp - fpm - fmp + fmm) / (4 * EPSILON ** 2)
            H[j, i] = H[i, j]
    return H

## Helper Functions: Muon Orthogonalization (Polar Factor)

The key operation that distinguishes Muon from SGD is the **orthogonalization** of the gradient matrix. Given a gradient matrix $G$ (the gradient of the loss with respect to a weight matrix), Muon replaces $G$ with its polar factor:

$$G = U \Sigma V^T \quad \Rightarrow \quad \text{ortho}(G) = U V^T$$

This is the closest orthogonal matrix to $G$ in the Frobenius norm. Geometrically, it preserves the *direction* of each singular component but sets all singular values to 1. This means:

1. **Direction information is preserved:** The principal axes of the gradient are maintained.
2. **Magnitude information is discarded:** All directions are weighted equally.
3. **The result is an orthogonal matrix:** $(\text{ortho}(G))^T \text{ortho}(G) = I$ (when $G$ is square and full-rank).

**Critical detail:** Muon orthogonalizes *each weight matrix's gradient separately*, then concatenates the results. So $W_1$'s gradient (a $2 \times 2$ matrix) is orthogonalized independently from $W_2$'s gradient (a $1 \times 2$ matrix). This per-layer structure is what might create the gauge-avoiding property we are testing.

In [ ]:
def muon_orthogonalize(G):
    """
    Newton-Schulz-style polar factor: ortho(G) = U @ V^T from SVD of G.
    This is the Muon update direction for a single weight matrix.
    For a matrix with fewer rows than columns (like W2: 1x2), we use
    the polar factor directly.
    """
    U, S, Vt = np.linalg.svd(G, full_matrices=False)
    return U @ Vt

## Helper Functions: SGD and Muon Update Computation

Below we define the two optimizer update rules side-by-side:

- **SGD update:** Simply $\Delta\theta = -\nabla L(\theta)$ (the negative gradient, unnormalized). The projection of this onto Hessian eigenvectors tells us how much the gradient itself aligns with each curvature direction.

- **Muon update:** Compute the gradient, reshape it into per-layer matrices, orthogonalize each via polar decomposition, negate, and concatenate back to a flat vector. The key question is whether this orthogonalization step *rotates* the update direction relative to the gradient in a way that avoids flat Hessian directions.

Note that for the projection analysis, we normalize both updates to unit vectors. So we are comparing *directions*, not magnitudes. This is the right comparison because we want to know *where* each optimizer points, not *how far* it steps.

In [ ]:
def compute_sgd_update(theta):
    """SGD update direction: -gradient (unnormalized)."""
    grad = compute_gradient(theta)
    return -grad

In [ ]:
def compute_muon_update(theta):
    """
    Muon update direction: for each weight matrix separately,
    compute gradient matrix, orthogonalize it via polar factor,
    then flatten all back to a 6-vector.
    """
    grad = compute_gradient(theta)
    # Split gradient into W1 and W2 parts
    grad_W1 = grad[:4].reshape(HIDDEN_DIM, INPUT_DIM)   # (2, 2)
    grad_W2 = grad[4:].reshape(OUTPUT_DIM, HIDDEN_DIM)  # (1, 2)

    # Orthogonalize each gradient matrix separately
    ortho_W1 = muon_orthogonalize(grad_W1)
    ortho_W2 = muon_orthogonalize(grad_W2)

    # Muon steps in the negative orthogonalized gradient direction
    update = np.concatenate([-ortho_W1.ravel(), -ortho_W2.ravel()])
    return update

## Helper Function: Projection onto Hessian Eigenvectors

The core measurement of this experiment. Given an update vector $\Delta\theta$ and the Hessian eigenvectors $\{v_1, \ldots, v_6\}$ (sorted by decreasing eigenvalue), we compute:

$$p_i = \left| \frac{\Delta\theta}{\|\Delta\theta\|} \cdot v_i \right|$$

This gives the absolute cosine of the angle between the update direction and each eigenvector. We take the absolute value because we only care about alignment, not sign (moving along $+v_i$ or $-v_i$ both correspond to motion along the $i$-th curvature direction).

**Interpretation of $p_i$:** Since the eigenvectors form an orthonormal basis, $\sum_i p_i^2 = 1$ (Parseval's theorem). So $p_i^2$ represents the *fraction of update energy* projected onto the $i$-th Hessian direction. If Muon avoids gauge directions, $p_5^2 + p_6^2$ (the gauge mass) should be small for Muon and larger for SGD.

In [ ]:
def project_onto_eigenvectors(update, eigvecs):
    """
    Project update onto each Hessian eigenvector.
    Returns |<update, v_i>| / ||update|| for each eigenvector.
    eigvecs: columns are eigenvectors (shape N_PARAMS x N_PARAMS).
    """
    norm = np.linalg.norm(update)
    if norm < 1e-12:
        return np.zeros(N_PARAMS)
    update_hat = update / norm
    projections = np.abs(eigvecs.T @ update_hat)  # each row is one eigenvector
    return projections

## Main Experiment Loop

The experiment proceeds as follows at each measurement step:

1. **Compute the full Hessian** $H(\theta)$ at the current parameters via finite differences.
2. **Eigendecompose** $H = Q \Lambda Q^T$ using `numpy.linalg.eigh` (which guarantees real eigenvalues for symmetric matrices). Sort eigenvectors by *decreasing* eigenvalue so that $v_1$ corresponds to the highest curvature (most "physical") direction and $v_6$ to the flattest (most "gauge-like") direction.
3. **Compute both updates:** The SGD update ($-\nabla L$) and the Muon update (orthogonalized per-layer gradients, negated).
4. **Project** each normalized update onto all 6 eigenvectors and record the absolute projections.
5. **Advance** the model by one SGD step (we use SGD for training; the Muon update is only computed for comparison, not applied).

The training loop runs for 200 steps total, with measurements at steps $\{0, 5, 10, 20, 40, 60, 80, 120, 160, 199\}$.

### Key Diagnostic Outputs

At each measurement step, we print:
- The full eigenvalue spectrum (to track how curvature evolves during training)
- Per-eigenvector projections for both SGD and Muon
- Sum-of-squared projections on "physical" (top-2) and "gauge" (bottom-2) eigenvectors
- The physical-to-gauge ratio for each optimizer

After all steps, we aggregate statistics and render a verdict.

In [ ]:
# ============================================================
# Main experiment
# ============================================================
def run_experiment():
    print("=" * 90)
    print("1.3b-ii-A: Hessian Top-5 Eigenvector Alignment With Update (2x2 deep linear net)")
    print("=" * 90)
    print(f"Network: {INPUT_DIM} -> {HIDDEN_DIM} -> {OUTPUT_DIM} (deep linear)")
    print(f"Total params: {N_PARAMS}")
    print(f"Data: {N_SAMPLES} samples, MSE loss")
    print(f"Hessian: full {N_PARAMS}x{N_PARAMS} via finite differences (eps={EPSILON})")
    print(f"Training steps: {N_TRAINING_STEPS}, measured at steps: {MEASURE_STEPS}")
    print()

    # Initialize weights
    W1 = np.random.randn(HIDDEN_DIM, INPUT_DIM) * 0.5
    W2 = np.random.randn(OUTPUT_DIM, HIDDEN_DIM) * 0.5
    theta = pack(W1, W2)

    print("=== Initial Weight Matrices ===")
    print(f"  W1 (hidden x input):\n{W1}")
    print(f"  W2 (output x hidden):\n{W2}")
    print(f"  Effective mapping W_eff = W2 @ W1:\n{W2 @ W1}")
    print(f"  Flattened theta: {theta}")
    print(f"  ||theta|| = {np.linalg.norm(theta):.6f}")
    print(f"  Initial loss: {loss_fn(theta):.6f}")
    print()

    # Storage for results across steps
    all_results = []

    # We train with plain SGD and measure at specific steps
    for step in range(N_TRAINING_STEPS):
        if step in MEASURE_STEPS:
            # -- Compute Hessian --
            H = compute_hessian(theta)
            eigenvalues, eigvecs = np.linalg.eigh(H)
            # eigh returns ascending order; we want descending
            idx = np.argsort(eigenvalues)[::-1]
            eigenvalues = eigenvalues[idx]
            eigvecs = eigvecs[:, idx]

            # -- Diagnostic: Hessian properties --
            print(f"--- Step {step:3d} | Loss = {loss_fn(theta):.6f} ---")
            print(f"  Hessian diagnostics:")
            print(f"    Trace(H) = {np.trace(H):.6f}  (sum of eigenvalues = {np.sum(eigenvalues):.6f})")
            print(f"    ||H||_F  = {np.linalg.norm(H, 'fro'):.6f}")
            print(f"    Condition number (lambda_1/lambda_6): ", end="")
            if abs(eigenvalues[-1]) > 1e-12:
                print(f"{abs(eigenvalues[0]/eigenvalues[-1]):.2f}")
            else:
                print(f"inf (lambda_6 ~ 0, gauge direction detected!)")
            print(f"    Eigenvalue spectrum: {eigenvalues}")
            print(f"    Eigenvalue ratio (max/min nonzero): ", end="")
            nonzero_evals = eigenvalues[np.abs(eigenvalues) > 1e-8]
            if len(nonzero_evals) > 0:
                print(f"{np.max(np.abs(nonzero_evals)) / np.min(np.abs(nonzero_evals)):.4f}")
            else:
                print("N/A (all eigenvalues ~ 0)")

            # -- Compute updates --
            sgd_update = compute_sgd_update(theta)
            muon_update = compute_muon_update(theta)

            # -- Diagnostic: update vectors --
            print(f"  Update diagnostics:")
            print(f"    SGD  update norm: {np.linalg.norm(sgd_update):.6f}")
            print(f"    Muon update norm: {np.linalg.norm(muon_update):.6f}")
            cosine_sgd_muon = np.dot(sgd_update, muon_update) / (np.linalg.norm(sgd_update) * np.linalg.norm(muon_update) + 1e-15)
            print(f"    Cosine(SGD, Muon): {cosine_sgd_muon:.6f}  (1.0 = identical directions, 0.0 = orthogonal)")
            angle_deg = np.degrees(np.arccos(np.clip(cosine_sgd_muon, -1, 1)))
            print(f"    Angle between SGD and Muon: {angle_deg:.2f} degrees")

            # -- Project onto eigenvectors --
            proj_sgd = project_onto_eigenvectors(sgd_update, eigvecs)
            proj_muon = project_onto_eigenvectors(muon_update, eigvecs)

            # -- Diagnostic: Parseval check --
            print(f"    Parseval check (should be 1.0): SGD={np.sum(proj_sgd**2):.6f}, Muon={np.sum(proj_muon**2):.6f}")

            current_loss = loss_fn(theta)

            result = {
                "step": step,
                "loss": current_loss,
                "eigenvalues": eigenvalues.copy(),
                "proj_sgd": proj_sgd.copy(),
                "proj_muon": proj_muon.copy(),
            }
            all_results.append(result)

            # -- Print table --
            print(f"  {'Eig#':>4s}  {'Eigenvalue':>12s}  {'|proj| SGD':>12s}  {'|proj| Muon':>12s}  {'Type':>10s}")
            print(f"  {'----':>4s}  {'----------':>12s}  {'----------':>12s}  {'-----------':>12s}  {'----':>10s}")

            # Label: top eigenvalues = physical, bottom = gauge (flat)
            for k in range(N_PARAMS):
                if k < 2:
                    label = "PHYSICAL"
                elif k >= N_PARAMS - 2:
                    label = "GAUGE"
                else:
                    label = "mid"
                print(f"  {k+1:4d}  {eigenvalues[k]:12.6f}  {proj_sgd[k]:12.6f}  {proj_muon[k]:12.6f}  {label:>10s}")

            # Summary stats
            phys_sgd = np.sum(proj_sgd[:2] ** 2)
            phys_muon = np.sum(proj_muon[:2] ** 2)
            gauge_sgd = np.sum(proj_sgd[-2:] ** 2)
            gauge_muon = np.sum(proj_muon[-2:] ** 2)
            print(f"  Sum-of-squares on PHYSICAL (top-2):  SGD={phys_sgd:.4f}  Muon={phys_muon:.4f}")
            print(f"  Sum-of-squares on GAUGE (bottom-2):  SGD={gauge_sgd:.4f}  Muon={gauge_muon:.4f}")
            ratio_sgd = phys_sgd / (gauge_sgd + 1e-15)
            ratio_muon = phys_muon / (gauge_muon + 1e-15)
            print(f"  Physical/Gauge ratio:                SGD={ratio_sgd:.4f}  Muon={ratio_muon:.4f}")
            if gauge_muon < gauge_sgd:
                print(f"  --> At this step, Muon has LESS gauge mass than SGD (ratio: {gauge_muon/(gauge_sgd+1e-15):.4f})")
            else:
                print(f"  --> At this step, Muon has MORE gauge mass than SGD (ratio: {gauge_muon/(gauge_sgd+1e-15):.4f})")
            print()

        # -- SGD training step (to advance the model) --
        grad = compute_gradient(theta)
        theta = theta - LR_SGD * grad

    # ============================================================
    # Aggregate analysis
    # ============================================================
    print("=" * 90)
    print("AGGREGATE ANALYSIS ACROSS ALL MEASURED STEPS")
    print("=" * 90)

    avg_proj_sgd = np.mean([r["proj_sgd"] for r in all_results], axis=0)
    avg_proj_muon = np.mean([r["proj_muon"] for r in all_results], axis=0)

    print(f"\n  Average |projection| across {len(all_results)} measurement steps:")
    print(f"  {'Eig#':>4s}  {'Avg |proj| SGD':>16s}  {'Avg |proj| Muon':>16s}  {'Muon/SGD ratio':>16s}  {'Type':>10s}")
    print(f"  {'----':>4s}  {'--------------':>16s}  {'---------------':>16s}  {'--------------':>16s}  {'----':>10s}")
    for k in range(N_PARAMS):
        if k < 2:
            label = "PHYSICAL"
        elif k >= N_PARAMS - 2:
            label = "GAUGE"
        else:
            label = "mid"
        ratio = avg_proj_muon[k] / (avg_proj_sgd[k] + 1e-15)
        print(f"  {k+1:4d}  {avg_proj_sgd[k]:16.6f}  {avg_proj_muon[k]:16.6f}  {ratio:16.4f}  {label:>10s}")

    # Physical vs Gauge mass (sum of squared projections, averaged)
    phys_mass_sgd = np.mean([np.sum(r["proj_sgd"][:2] ** 2) for r in all_results])
    phys_mass_muon = np.mean([np.sum(r["proj_muon"][:2] ** 2) for r in all_results])
    gauge_mass_sgd = np.mean([np.sum(r["proj_sgd"][-2:] ** 2) for r in all_results])
    gauge_mass_muon = np.mean([np.sum(r["proj_muon"][-2:] ** 2) for r in all_results])

    print(f"\n  Average sum-of-squared projections:")
    print(f"    PHYSICAL (top-2):   SGD={phys_mass_sgd:.6f}   Muon={phys_mass_muon:.6f}")
    print(f"    GAUGE (bottom-2):   SGD={gauge_mass_sgd:.6f}   Muon={gauge_mass_muon:.6f}")
    print(f"    Physical/Gauge:     SGD={phys_mass_sgd/(gauge_mass_sgd+1e-15):.4f}   Muon={phys_mass_muon/(gauge_mass_muon+1e-15):.4f}")

    # Additional diagnostic: standard deviation across steps
    print(f"\n  Variability across steps (std of sum-of-squared projections):")
    phys_std_sgd = np.std([np.sum(r["proj_sgd"][:2] ** 2) for r in all_results])
    phys_std_muon = np.std([np.sum(r["proj_muon"][:2] ** 2) for r in all_results])
    gauge_std_sgd = np.std([np.sum(r["proj_sgd"][-2:] ** 2) for r in all_results])
    gauge_std_muon = np.std([np.sum(r["proj_muon"][-2:] ** 2) for r in all_results])
    print(f"    PHYSICAL std:  SGD={phys_std_sgd:.6f}   Muon={phys_std_muon:.6f}")
    print(f"    GAUGE std:     SGD={gauge_std_sgd:.6f}   Muon={gauge_std_muon:.6f}")

    # Hypothesis test: does Muon have lower gauge projection than SGD?
    gauge_proj_sgd_all = [np.sum(r["proj_sgd"][-2:] ** 2) for r in all_results]
    gauge_proj_muon_all = [np.sum(r["proj_muon"][-2:] ** 2) for r in all_results]
    muon_less_gauge = sum(1 for s, m in zip(gauge_proj_sgd_all, gauge_proj_muon_all) if m < s)

    print(f"\n  Hypothesis check: Muon has less gauge mass than SGD?")
    print(f"    Steps where Muon gauge-mass < SGD gauge-mass: {muon_less_gauge}/{len(all_results)}")

    phys_proj_sgd_all = [np.sum(r["proj_sgd"][:2] ** 2) for r in all_results]
    phys_proj_muon_all = [np.sum(r["proj_muon"][:2] ** 2) for r in all_results]
    muon_more_phys = sum(1 for s, m in zip(phys_proj_sgd_all, phys_proj_muon_all) if m > s)
    print(f"    Steps where Muon physical-mass > SGD physical-mass: {muon_more_phys}/{len(all_results)}")

    # Per-step breakdown for transparency
    print(f"\n  Per-step gauge mass comparison:")
    for i, r in enumerate(all_results):
        g_sgd = np.sum(r["proj_sgd"][-2:] ** 2)
        g_muon = np.sum(r["proj_muon"][-2:] ** 2)
        winner = "Muon < SGD" if g_muon < g_sgd else "SGD < Muon"
        print(f"    Step {r['step']:3d}: SGD_gauge={g_sgd:.4f}, Muon_gauge={g_muon:.4f}  ({winner})")

    # Eigenvalue spectrum summary
    print(f"\n  Eigenvalue spectrum across steps (showing spread):")
    for k in range(N_PARAMS):
        evals_k = [r["eigenvalues"][k] for r in all_results]
        if k < 2:
            label = "PHYSICAL"
        elif k >= N_PARAMS - 2:
            label = "GAUGE"
        else:
            label = "mid"
        print(f"    Eig {k+1}: mean={np.mean(evals_k):10.6f}  std={np.std(evals_k):10.6f}  min={np.min(evals_k):10.6f}  max={np.max(evals_k):10.6f}  [{label}]")

    # Check if bottom eigenvalues are truly near-zero (gauge)
    bottom_evals = [r["eigenvalues"][-1] for r in all_results]
    print(f"\n  Gauge direction verification:")
    print(f"    Smallest eigenvalue across steps: mean={np.mean(bottom_evals):.6f}, max={np.max(np.abs(bottom_evals)):.6f}")
    if np.max(np.abs(bottom_evals)) < 0.01:
        print(f"    --> Confirmed: bottom eigenvalue is near-zero (true gauge direction)")
    else:
        print(f"    --> Note: bottom eigenvalue is not negligible; gauge symmetry is only approximate at these parameter values")

    # ============================================================
    # Verdict
    # ============================================================
    print("\n" + "=" * 90)
    print("VERDICT")
    print("=" * 90)

    avg_gauge_ratio = gauge_mass_muon / (gauge_mass_sgd + 1e-15)
    avg_phys_ratio = phys_mass_muon / (phys_mass_sgd + 1e-15)

    if avg_gauge_ratio < 0.8 and muon_less_gauge >= len(all_results) * 0.7:
        print("  CONFIRMED: Muon consistently avoids gauge (flat Hessian) directions.")
        print(f"  Muon's gauge mass is {avg_gauge_ratio:.2f}x that of SGD (averaged).")
        verdict = "CONFIRMED"
    elif avg_gauge_ratio > 1.2 and muon_less_gauge <= len(all_results) * 0.3:
        print("  REFUTED: Muon has MORE mass on gauge directions than SGD.")
        print(f"  Muon's gauge mass is {avg_gauge_ratio:.2f}x that of SGD (averaged).")
        verdict = "REFUTED"
    else:
        print("  NUANCED: Muon's avoidance of gauge directions is not clear-cut.")
        print(f"  Muon's gauge mass is {avg_gauge_ratio:.2f}x that of SGD.")
        print(f"  Muon's physical mass is {avg_phys_ratio:.2f}x that of SGD.")
        print(f"  Steps with Muon < SGD on gauge: {muon_less_gauge}/{len(all_results)}")
        verdict = "NUANCED"

    print(f"\n  Context connection:")
    print(f"  - 1.2b-i found Muon does NOT distinguish gauge/physical in Lyapunov terms.")
    print(f"  - 1.3b-i-A found Muon REVERSES the feedback loop (corr = -0.51).")
    if verdict == "CONFIRMED":
        print(f"  - This result shows the mechanism: Muon's orthogonalization steers")
        print(f"    the update away from flat (gauge) Hessian directions.")
    elif verdict == "NUANCED":
        print(f"  - The Hessian alignment picture is nuanced, consistent with 1.2b-i's")
        print(f"    finding that Muon doesn't cleanly separate gauge from physical.")
        print(f"    The orthogonalization affects direction but not in a simple")
        print(f"    gauge-avoiding way.")
    print("=" * 90)

    # ============================================================
    # Plot
    # ============================================================
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        n_steps = len(all_results)
        fig, axes = plt.subplots(2, 5, figsize=(22, 8), sharey=True)
        axes = axes.flatten()

        for idx_r, r in enumerate(all_results):
            ax = axes[idx_r]
            x_pos = np.arange(N_PARAMS)
            width = 0.35
            bars_sgd = ax.bar(x_pos - width / 2, r["proj_sgd"], width,
                              color="steelblue", alpha=0.8, label="SGD")
            bars_muon = ax.bar(x_pos + width / 2, r["proj_muon"], width,
                               color="firebrick", alpha=0.8, label="Muon")

            ax.set_title(f"Step {r['step']}\nLoss={r['loss']:.4f}", fontsize=9)
            ax.set_xlabel("Eigenvector (1=top)", fontsize=8)
            ax.set_xticks(x_pos)
            ax.set_xticklabels([f"{k+1}\n({r['eigenvalues'][k]:.3f})"
                                for k in range(N_PARAMS)], fontsize=7)

            # Shade gauge region
            ax.axvspan(N_PARAMS - 2 - 0.5, N_PARAMS - 0.5, alpha=0.1,
                        color="gray", label="gauge zone")

            if idx_r == 0:
                ax.legend(fontsize=7, loc="upper right")

        axes[0].set_ylabel("|projection| / ||update||", fontsize=9)
        axes[5].set_ylabel("|projection| / ||update||", fontsize=9)

        fig.suptitle("1.3b-ii-A: Hessian Eigenvector Alignment -- SGD vs Muon\n"
                     "(Eigenvalues descending: left=physical/high-curvature, right=gauge/flat)",
                     fontsize=12, fontweight="bold")
        plt.tight_layout(rect=[0, 0, 1, 0.93])

        plot_path = os.path.join(SCRIPT_DIR, "hessian_eigenvector_alignment.png")
        plt.savefig(plot_path, dpi=150)
        print(f"\n  Plot saved to: {plot_path}")
        plt.close()

        # --- Additional summary plot: average projections ---
        fig2, ax2 = plt.subplots(figsize=(8, 5))
        x_pos = np.arange(N_PARAMS)
        width = 0.35
        ax2.bar(x_pos - width / 2, avg_proj_sgd, width,
                color="steelblue", alpha=0.85, label="SGD (avg)")
        ax2.bar(x_pos + width / 2, avg_proj_muon, width,
                color="firebrick", alpha=0.85, label="Muon (avg)")

        avg_evals = np.mean([r["eigenvalues"] for r in all_results], axis=0)
        ax2.set_xticks(x_pos)
        ax2.set_xticklabels([f"v{k+1}\n(lam={avg_evals[k]:.3f})"
                             for k in range(N_PARAMS)], fontsize=8)
        ax2.set_ylabel("|projection| / ||update||")
        ax2.set_xlabel("Hessian eigenvector (1=highest curvature, 6=flattest)")
        ax2.set_title("Average Eigenvector Alignment: SGD vs Muon\n"
                       f"(across {n_steps} steps, {verdict})")
        ax2.axvspan(N_PARAMS - 2 - 0.5, N_PARAMS - 0.5, alpha=0.12,
                     color="gray", label="gauge zone")
        ax2.legend()
        plt.tight_layout()

        plot2_path = os.path.join(SCRIPT_DIR, "hessian_alignment_summary.png")
        fig2.savefig(plot2_path, dpi=150)
        print(f"  Summary plot saved to: {plot2_path}")
        plt.close()

    except ImportError:
        print("\n  [matplotlib not available; skipping plots]")

    return all_results

## Execution

Run the full experiment below. The output includes:
1. **Per-step tables** showing eigenvalue spectra and projection magnitudes for both SGD and Muon
2. **Hessian diagnostics** (trace, Frobenius norm, condition number) to verify numerical health
3. **Update diagnostics** (norms, cosine similarity between SGD and Muon directions)
4. **Parseval checks** confirming that projections sum to 1 (numerical sanity)
5. **Aggregate statistics** across all 10 measurement steps
6. **Per-step gauge mass comparison** for full transparency
7. **Eigenvalue spectrum evolution** to track how curvature changes during training
8. **Verdict** with connection to prior experiments in this series

In [ ]:
if __name__ == "__main__":
    run_experiment()

## Conclusions and Interpretation

### What This Experiment Measures

This experiment directly tests a specific geometric prediction of the "Muon as gauge-fixing" hypothesis: that Muon's polar decomposition of the gradient matrix should naturally avoid stepping along flat (gauge) directions of the Hessian. The test is clean because:

1. We use the **exact same gradient** for both SGD and Muon -- the only difference is the orthogonalization step.
2. We compare **normalized directions**, removing any confound from step size differences.
3. We use the **full Hessian** (not an approximation), so the eigendecomposition is exact up to finite-difference error.

### How to Read the Results

- **Physical/Gauge ratio > 1** means the optimizer concentrates more on high-curvature directions (good for optimization efficiency).
- **Muon gauge mass / SGD gauge mass < 1** means Muon is indeed avoiding flat directions more than SGD.
- The **cosine similarity** between SGD and Muon updates tells us how much the orthogonalization actually changes the direction -- if it is close to 1.0, the orthogonalization has little effect on direction.

### Connection to the Broader Research Program

This experiment sits within a chain of evidence:

- **1.2b-i (Lyapunov):** Found that Muon does NOT distinguish gauge from physical in terms of trajectory sensitivity. This suggests Muon's gauge-fixing (if any) is not through trajectory stability.
- **1.3b-i-A (Feedback loop):** Found that Muon *reverses* the correlation between gradient singular values and weight singular values (corr = -0.51 vs +0.49 for SGD). This suggests Muon's orthogonalization creates a *negative feedback* that resists spectral concentration.
- **This experiment (1.3b-ii-A):** Tests whether the mechanism operates at the level of Hessian eigenvector alignment -- does the orthogonalization geometrically rotate the update away from flat directions?

### Caveats

1. **Small network:** With only 6 parameters, the gauge structure is very constrained. Results may not generalize to larger networks where gauge directions are more numerous and complex.
2. **Deep linear network:** The gauge symmetry is exact only for linear networks. Nonlinear activations break the exact symmetry, though approximate gauge directions may persist.
3. **Training with SGD:** The model is trained with SGD throughout; Muon updates are computed but not applied. This means we are comparing what Muon *would do* at SGD-trained parameters, not what happens when Muon actually trains the network.
4. **Fixed labeling:** We label the top-2 eigenvectors as "physical" and bottom-2 as "gauge" based solely on eigenvalue magnitude. In principle, the gauge directions could have non-trivial eigenvalues away from exact symmetry points.